In [ ]:
"""
Transaction-Level Data EDA — V3
================================
Comprehensive EDA for 188-column enriched AML transaction dataset.
Covers all feature groups:
  1.  Overview KPIs & Null Audit
  2.  Transaction Characteristics
  3.  Temporal Features
  4.  Customer & Account Profile
  5.  Device & Network Signals
  6.  Beneficiary Profile
  7.  IP / Geo Features
  8.  Flow / Graph Features
  9.  Shared Identity Signals
 10.  Sender Account Velocity  (1h/24h/7d/30d)
 11.  Sender Customer Velocity (1h/24h/7d/30d)
 12.  Sender Balance Features
 13.  Receiver Account Velocity
 14.  Receiver Balance Features
 15.  AML Rule Engine
 16.  Rule Lift Analysis (Fraud vs Legit)
 17.  Fraud Intensity Score (FIS)
 18.  Fraud Type Examples
 19.  Fraud Transaction Wide Table
"""

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pathlib import Path

BASE_DIR     = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent

INPUT_PATH  = PROJECT_ROOT / "outputs" / "transactions_enriched.parquet"
OUTPUT_PATH = PROJECT_ROOT / "EDA" / "transactions_summary_V3.xlsx"

print(f"Project root : {PROJECT_ROOT}")
print(f"Input        : {INPUT_PATH}")
print(f"Output       : {OUTPUT_PATH}")

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
if INPUT_PATH.suffix == ".parquet":
    df = pd.read_parquet(INPUT_PATH)
else:
    df = pd.read_csv(INPUT_PATH, low_memory=False)

# Parse timestamp if needed
if "timestamp" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
if "txn_date" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["txn_date"]):
    df["txn_date"] = pd.to_datetime(df["txn_date"], errors="coerce")

TOTAL = len(df)

# Cast flow/graph numeric columns that may be stored as object
for _col in ["flow_depth", "hop_number", "time_since_origin_ts"]:
    if _col in df.columns and not pd.api.types.is_numeric_dtype(df[_col]):
        df[_col] = pd.to_numeric(df[_col], errors="coerce")

print(f"✅ Loaded {TOTAL:,} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns[:10])} ...")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STYLING & HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

HDR_FILL  = PatternFill("solid", start_color="1F3864")
SUB_FILL  = PatternFill("solid", start_color="2E75B6")
HDR2_FILL = PatternFill("solid", start_color="BDD7EE")
ALT_FILL  = PatternFill("solid", start_color="D9E1F2")
WHITE     = PatternFill("solid", start_color="FFFFFF")
RED_FILL  = PatternFill("solid", start_color="FFE0E0")
WARN_FILL = PatternFill("solid", start_color="FFF2CC")
GREEN_FILL= PatternFill("solid", start_color="E2EFDA")
THIN = Border(
    left=Side(style="thin", color="B8CCE4"),
    right=Side(style="thin", color="B8CCE4"),
    top=Side(style="thin", color="B8CCE4"),
    bottom=Side(style="thin", color="B8CCE4"),
)

def hdr(ws, row, col, val, size=11, fill=HDR_FILL, color="FFFFFF"):
    c = ws.cell(row=row, column=col, value=val)
    c.font      = Font(name="Arial", bold=True, size=size, color=color)
    c.fill      = fill
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = THIN
    return c

def dat(ws, row, col, val, alt=False, bold=False, center=False, red=False, green=False, warn=False):
    c = ws.cell(row=row, column=col, value=val)
    c.font      = Font(name="Arial", size=10, bold=bold, color="1F2D3D")
    fill = RED_FILL if red else (GREEN_FILL if green else (WARN_FILL if warn else (ALT_FILL if alt else WHITE)))
    c.fill      = fill
    c.alignment = Alignment(horizontal="center" if center else "left", vertical="center")
    c.border    = THIN
    return c

def write_block(ws, sr, sc, title, dataframe, flag_high=None):
    """Write a titled table block. Returns next available row."""
    if dataframe is None or (hasattr(dataframe, "empty") and dataframe.empty):
        return sr
    ec = sc + len(dataframe.columns) - 1
    ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
    hdr(ws, sr, sc, title, size=10, fill=SUB_FILL)
    ws.row_dimensions[sr].height = 22
    for ci, col_name in enumerate(dataframe.columns, start=sc):
        hdr(ws, sr+1, ci, col_name, size=9, fill=HDR2_FILL, color="1F2D3D")
    for ri, (_, row_data) in enumerate(dataframe.iterrows()):
        alt = ri % 2 == 1
        for ci, val in enumerate(row_data, start=sc):
            is_red = False
            if flag_high and dataframe.columns[ci - sc] == "Percentage (%)" and isinstance(val, float) and val > flag_high:
                is_red = True
            dat(ws, sr+2+ri, ci, val, alt=alt, center=(ci > sc), red=is_red)
    for ci in range(sc, ec+1):
        cl = get_column_letter(ci)
        max_w = max(len(str(ws.cell(r, ci).value or "")) for r in range(sr, sr+2+len(dataframe)))
        ws.column_dimensions[cl].width = min(max(max_w+4, 14), 50)
    return sr + 2 + len(dataframe) + 2

# ── Summary helpers ───────────────────────────────────────────────────────────

def pct_df(series, label, top_n=None):
    d = series.value_counts().reset_index()
    d.columns = [label, "Count"]
    d["Percentage (%)"] = (d["Count"] / TOTAL * 100).round(2)
    return d.head(top_n) if top_n else d

def stats_df(series, label):
    """Describe a series. Coerces to numeric if needed; falls back to value_counts for object dtype."""
    if not pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        if numeric.notna().sum() > 0:
            series = numeric   # use the coerced numeric version
        else:
            # Truly non-numeric (e.g. timedeltas stored as strings) — show value_counts summary
            vc = series.value_counts().head(8)
            d = pd.DataFrame({"Statistic": list(vc.index), label: list(vc.values)})
            return d
    d = series.describe().round(4).reset_index()
    d.columns = ["Statistic", label]
    return d

def flag_summary(series, label, mapping=None):
    d = series.value_counts().reset_index()
    d.columns = [label, "Count"]
    if mapping:
        d[label] = d[label].map(mapping).fillna(d[label])
    d["Percentage (%)"] = (d["Count"] / TOTAL * 100).round(2)
    return d

def fraud_legit_compare(series, label):
    """Compare categorical feature fraud vs legit rate."""
    if "label" not in df.columns:
        return pd.DataFrame()
    tmp = df.assign(_cat=series)
    grp = tmp.groupby(["_cat", "label"]).size().unstack(fill_value=0)
    if 0 not in grp.columns: grp[0] = 0
    if 1 not in grp.columns: grp[1] = 0
    grp.columns = ["Legit", "Fraud"]
    grp["Total"] = grp["Legit"] + grp["Fraud"]
    grp["Fraud Rate (%)"] = (grp["Fraud"] / grp["Total"] * 100).round(2)
    return grp.reset_index().rename(columns={"_cat": label}).sort_values("Fraud Rate (%)", ascending=False)

def velocity_stats_table(prefix_cols):
    cols = [c for c in prefix_cols if c in df.columns]
    if not cols: return pd.DataFrame()
    t = df[cols].describe().round(2).T.reset_index()
    t.columns = ["Feature"] + list(t.columns[1:])
    return t

def fraud_vs_legit_stats(cols_list):
    if "label" not in df.columns: return pd.DataFrame()
    cols = [c for c in cols_list if c in df.columns]
    if not cols: return pd.DataFrame()
    fraud = df[df["label"]==1][cols].mean().round(4)
    legit = df[df["label"]==0][cols].mean().round(4)
    out = pd.DataFrame({"Feature": cols, "Fraud Mean": fraud.values, "Legit Mean": legit.values})
    out["Ratio (F/L)"] = (fraud.values / (legit.values + 1e-9)).round(3)
    return out.sort_values("Ratio (F/L)", ascending=False).reset_index(drop=True)

def rule_summary_fn(rule_cols_list):
    rows = []
    for col in rule_cols_list:
        if col not in df.columns: continue
        fired = int(df[col].sum())
        fraud_rate = round(df[df["label"]==1][col].mean()*100, 2) if "label" in df.columns else None
        legit_rate = round(df[df["label"]==0][col].mean()*100, 2) if "label" in df.columns else None
        rows.append({
            "Rule Name": col.replace("rule_","").replace("_"," ").title(),
            "Column": col,
            "Times Fired": fired,
            "Fire Rate (%)": round(fired / TOTAL * 100, 2),
            "Fraud Fire (%)": fraud_rate,
            "Legit Fire (%)": legit_rate,
        })
    return pd.DataFrame(rows).sort_values("Times Fired", ascending=False).reset_index(drop=True)

print("✅ Helpers defined")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — OVERVIEW KPIs
# ══════════════════════════════════════════════════════════════════════════════
fraud_count   = int(df["label"].sum()) if "label" in df.columns else 0
legit_count   = TOTAL - fraud_count
fraud_pct     = round(fraud_count / TOTAL * 100, 2)
total_amount  = df["amount"].sum() if "amount" in df.columns else 0
avg_amount    = df["amount"].mean() if "amount" in df.columns else 0
median_amount = df["amount"].median() if "amount" in df.columns else 0
max_amount    = df["amount"].max() if "amount" in df.columns else 0

overview_kpis = pd.DataFrame({
    "KPI": [
        "Total Transactions", "Fraud Transactions", "Legitimate Transactions", "Fraud Rate (%)",
        "Total Amount (INR)", "Avg Amount (INR)", "Median Amount (INR)", "Max Amount (INR)",
        "Unique Customers", "Unique Sender Accounts", "Unique Receiver Accounts",
        "Unique Devices", "Unique Beneficiaries",
        "Date Range Start", "Date Range End", "Total Columns",
    ],
    "Value": [
        f"{TOTAL:,}", f"{fraud_count:,}", f"{legit_count:,}", f"{fraud_pct}%",
        f"₹{total_amount:,.2f}", f"₹{avg_amount:,.2f}", f"₹{median_amount:,.2f}", f"₹{max_amount:,.2f}",
        df["customer_id"].nunique() if "customer_id" in df.columns else "N/A",
        df["sender_account_id"].nunique() if "sender_account_id" in df.columns else "N/A",
        df["receiver_account_id"].nunique() if "receiver_account_id" in df.columns else "N/A",
        df["device_id"].nunique() if "device_id" in df.columns else "N/A",
        df["beneficiary_id"].nunique() if "beneficiary_id" in df.columns else "N/A",
        str(df["timestamp"].min())[:10] if "timestamp" in df.columns else "N/A",
        str(df["timestamp"].max())[:10] if "timestamp" in df.columns else "N/A",
        df.shape[1],
    ]
})

fraud_type_df = pct_df(df["fraud_type"], "Fraud Type") if "fraud_type" in df.columns else pd.DataFrame()

null_audit = pd.DataFrame({
    "Column": df.columns,
    "Null Count": df.isnull().sum().values,
    "Null %": (df.isnull().sum().values / TOTAL * 100).round(2),
    "Dtype": df.dtypes.astype(str).values,
}).query("`Null Count` > 0").sort_values("Null %", ascending=False).reset_index(drop=True)

print(f"Fraud rate: {fraud_pct}%  |  Columns with nulls: {len(null_audit)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — TRANSACTION CHARACTERISTICS
# ══════════════════════════════════════════════════════════════════════════════
amount_stats   = stats_df(df["amount"], "Amount (INR)") if "amount" in df.columns else pd.DataFrame()
channel_df     = pct_df(df["channel"], "Channel") if "channel" in df.columns else pd.DataFrame()
txn_type_df    = pct_df(df["transaction_type"], "Transaction Type") if "transaction_type" in df.columns else pd.DataFrame()
debit_df       = flag_summary(df["debit_credit"], "Debit/Credit") if "debit_credit" in df.columns else pd.DataFrame()
cash_df        = flag_summary(df["cash_flag"], "Cash Flag", {0:"Non-Cash",1:"Cash"}) if "cash_flag" in df.columns else pd.DataFrame()
dormancy_df    = flag_summary(df["dormancy_flag"], "Dormancy Flag", {0:"Active",1:"Dormant/Reactivated"}) if "dormancy_flag" in df.columns else pd.DataFrame()

channel_fraud  = fraud_legit_compare(df["channel"], "Channel") if "channel" in df.columns and "label" in df.columns else pd.DataFrame()
txn_type_fraud = fraud_legit_compare(df["transaction_type"], "Transaction Type") if "transaction_type" in df.columns and "label" in df.columns else pd.DataFrame()

if "dormancy_flag" in df.columns and "sender_account_id" in df.columns:
    total_accts  = df["sender_account_id"].nunique()
    dormant_accts= df[df["dormancy_flag"]==1]["sender_account_id"].nunique()
    dormant_acct_df = pd.DataFrame({
        "Account Status": ["Dormant (≥1 reactivation txn)", "Active", "Total Unique Accounts"],
        "Account Count":  [dormant_accts, total_accts-dormant_accts, total_accts],
        "Percentage (%)": [round(dormant_accts/total_accts*100,2), round((total_accts-dormant_accts)/total_accts*100,2), 100.0],
    })
else:
    dormant_acct_df = pd.DataFrame()

print("✅ Section 2 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — TEMPORAL FEATURES
# ══════════════════════════════════════════════════════════════════════════════
hour_df    = pct_df(df["txn_hour"], "Hour of Day") if "txn_hour" in df.columns else pd.DataFrame()
dow_map    = {0:"Monday",1:"Tuesday",2:"Wednesday",3:"Thursday",4:"Friday",5:"Saturday",6:"Sunday"}
dow_df     = pct_df(df["txn_day_of_week"].map(dow_map), "Day of Week") if "txn_day_of_week" in df.columns else pd.DataFrame()
month_map  = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
month_df   = pct_df(df["txn_month"].map(month_map), "Month") if "txn_month" in df.columns else pd.DataFrame()
quarter_df = pct_df(df["txn_quarter"], "Quarter") if "txn_quarter" in df.columns else pd.DataFrame()
dom_df     = pct_df(df["txn_day_of_month"], "Day of Month") if "txn_day_of_month" in df.columns else pd.DataFrame()
night_df   = flag_summary(df["is_night"], "Is Night", {0:"Day",1:"Night (22–05h)"}) if "is_night" in df.columns else pd.DataFrame()
weekend_df = flag_summary(df["is_weekend"], "Is Weekend", {0:"Weekday",1:"Weekend"}) if "is_weekend" in df.columns else pd.DataFrame()
biz_df     = flag_summary(df["is_business_hours"], "Business Hours", {0:"Non-Business",1:"Business (09–17h M–F)"}) if "is_business_hours" in df.columns else pd.DataFrame()
early_df   = flag_summary(df["is_early_morning"], "Early Morning", {0:"No",1:"Yes (05–09h)"}) if "is_early_morning" in df.columns else pd.DataFrame()

if "label" in df.columns:
    temp_rows = []
    for col, txt in [("is_night","Night (22–05h)"),("is_weekend","Weekend"),
                     ("is_business_hours","Business Hours"),("is_early_morning","Early Morning (05–09h)")]:
        if col in df.columns:
            sub = df[df[col]==1]
            temp_rows.append({"Time Window": txt, "Total Txns": len(sub),
                               "Fraud Count": int(sub["label"].sum()),
                               "Fraud Rate (%)": round(sub["label"].mean()*100, 2) if len(sub)>0 else 0})
    temporal_fraud_df = pd.DataFrame(temp_rows)
else:
    temporal_fraud_df = pd.DataFrame()

print("✅ Section 3 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — CUSTOMER & ACCOUNT PROFILE
# ══════════════════════════════════════════════════════════════════════════════
occ_df        = pct_df(df["occupation"], "Occupation") if "occupation" in df.columns else pd.DataFrame()
kyc_df        = pct_df(df["kyc_level"], "KYC Level") if "kyc_level" in df.columns else pd.DataFrame()
crisk_df      = pct_df(df["country_risk"], "Country Risk") if "country_risk" in df.columns else pd.DataFrame()
inc_df        = pct_df(df["income_bracket"], "Income Bracket") if "income_bracket" in df.columns else pd.DataFrame()
cust_risk_df  = pct_df(df["customer_risk_rating"], "Customer Risk Rating") if "customer_risk_rating" in df.columns else pd.DataFrame()
pep_df        = flag_summary(df["pep_flag"], "PEP Flag", {0:"Non-PEP",1:"PEP"}) if "pep_flag" in df.columns else pd.DataFrame()
industry_df   = pct_df(df["industry"], "Industry") if "industry" in df.columns else pd.DataFrame()
acc_type_df   = pct_df(df["account_type"], "Account Type") if "account_type" in df.columns else pd.DataFrame()
avg_bal_stats = stats_df(df["avg_balance"], "Avg Balance (INR)") if "avg_balance" in df.columns else pd.DataFrame()
acct_age_stats= stats_df(df["account_open_days"], "Account Age (Days)") if "account_open_days" in df.columns else pd.DataFrame()
crr_num_stats = stats_df(df["customer_risk_rating_num"], "CRR (Numeric)") if "customer_risk_rating_num" in df.columns else pd.DataFrame()
kyc_num_stats = stats_df(df["kyc_level_num"], "KYC Level (Numeric)") if "kyc_level_num" in df.columns else pd.DataFrame()

kyc_fraud     = fraud_legit_compare(df["kyc_level"], "KYC Level") if "kyc_level" in df.columns and "label" in df.columns else pd.DataFrame()
crisk_fraud   = fraud_legit_compare(df["country_risk"], "Country Risk") if "country_risk" in df.columns and "label" in df.columns else pd.DataFrame()
inc_fraud     = fraud_legit_compare(df["income_bracket"], "Income Bracket") if "income_bracket" in df.columns and "label" in df.columns else pd.DataFrame()
crr_fraud     = fraud_legit_compare(df["customer_risk_rating"], "Customer Risk Rating") if "customer_risk_rating" in df.columns and "label" in df.columns else pd.DataFrame()
occ_fraud     = fraud_legit_compare(df["occupation"], "Occupation") if "occupation" in df.columns and "label" in df.columns else pd.DataFrame()

print("✅ Section 4 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — DEVICE & NETWORK SIGNALS
# ══════════════════════════════════════════════════════════════════════════════
os_df        = pct_df(df["os_type"], "OS Type") if "os_type" in df.columns else pd.DataFrame()
rooted_df    = flag_summary(df["rooted_flag"], "Rooted Device", {0:"Not Rooted",1:"Rooted"}) if "rooted_flag" in df.columns else pd.DataFrame()
vpn_df       = flag_summary(df["vpn_flag"], "VPN/Proxy", {0:"No VPN",1:"VPN Active"}) if "vpn_flag" in df.columns else pd.DataFrame()
emu_df       = flag_summary(df["emulator_flag"], "Emulator", {0:"Real Device",1:"Emulator"}) if "emulator_flag" in df.columns else pd.DataFrame()
dev_age_stats= stats_df(df["device_age_days"], "Device Age (Days)") if "device_age_days" in df.columns else pd.DataFrame()

if "label" in df.columns:
    dev_rows = []
    for col, txt in [("rooted_flag","Rooted Device"),("vpn_flag","VPN Active"),("emulator_flag","Emulator")]:
        if col in df.columns:
            sub = df[df[col]==1]
            dev_rows.append({"Device Signal": txt, "Total Txns": len(sub),
                              "Fraud Count": int(sub["label"].sum()),
                              "Fraud Rate (%)": round(sub["label"].mean()*100, 2) if len(sub)>0 else 0})
    device_fraud_df = pd.DataFrame(dev_rows)
else:
    device_fraud_df = pd.DataFrame()

print("✅ Section 5 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — BENEFICIARY PROFILE
# ══════════════════════════════════════════════════════════════════════════════
ben_type_df   = pct_df(df["beneficiary_type"], "Beneficiary Type") if "beneficiary_type" in df.columns else pd.DataFrame()
ben_risk_df   = pct_df(df["beneficiary_country_risk"], "Beneficiary Country Risk") if "beneficiary_country_risk" in df.columns else pd.DataFrame()
ben_fraud     = fraud_legit_compare(df["beneficiary_type"], "Beneficiary Type") if "beneficiary_type" in df.columns and "label" in df.columns else pd.DataFrame()
benrisk_fraud = fraud_legit_compare(df["beneficiary_country_risk"], "Bene Country Risk") if "beneficiary_country_risk" in df.columns and "label" in df.columns else pd.DataFrame()
print("✅ Section 6 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — IP / GEO FEATURES
# ══════════════════════════════════════════════════════════════════════════════
ip_risk_stats = stats_df(df["ip_risk_score"], "IP Risk Score") if "ip_risk_score" in df.columns else pd.DataFrame()
geo_lat_stats = stats_df(df["geo_lat"], "Geo Latitude") if "geo_lat" in df.columns else pd.DataFrame()
geo_lon_stats = stats_df(df["geo_lon"], "Geo Longitude") if "geo_lon" in df.columns else pd.DataFrame()
home_city_df  = pct_df(df["home_city"], "Home City", top_n=20) if "home_city" in df.columns else pd.DataFrame()

if "ip_risk_score" in df.columns:
    bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
    labels_b = ["Very Low (0–0.2)","Low (0.2–0.4)","Medium (0.4–0.6)","High (0.6–0.8)","Very High (0.8–1.0)"]
    bucket = pd.cut(df["ip_risk_score"], bins=bins, labels=labels_b, right=False)
    ip_bucket_df = pct_df(bucket, "IP Risk Band")
    if "label" in df.columns:
        ip_fraud_rows = []
        for band in labels_b:
            sub = df[bucket==band]
            if len(sub) > 0:
                ip_fraud_rows.append({"IP Risk Band": band, "Txn Count": len(sub),
                                       "Fraud Count": int(sub["label"].sum()),
                                       "Fraud Rate (%)": round(sub["label"].mean()*100, 2)})
        ip_fraud_df = pd.DataFrame(ip_fraud_rows)
    else:
        ip_fraud_df = pd.DataFrame()
else:
    ip_bucket_df = pd.DataFrame()
    ip_fraud_df  = pd.DataFrame()

print("✅ Section 7 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — FLOW / GRAPH FEATURES
# ══════════════════════════════════════════════════════════════════════════════
if "synthetic_flow_id" in df.columns:
    has_flow = df["synthetic_flow_id"].notna() & (df["synthetic_flow_id"].astype(str).str.strip() != "")
    flow_summary = pd.DataFrame({
        "Status": ["Has Flow ID (mule_ring/layering)", "No Flow ID"],
        "Count":  [int(has_flow.sum()), int((~has_flow).sum())],
        "Percentage (%)": [round(has_flow.mean()*100, 2), round((~has_flow).mean()*100, 2)],
    })
else:
    flow_summary = pd.DataFrame()

flow_depth_stats  = stats_df(df["flow_depth"], "Flow Depth") if "flow_depth" in df.columns else pd.DataFrame()
hop_stats         = stats_df(df["hop_number"], "Hop Number") if "hop_number" in df.columns else pd.DataFrame()
time_origin_stats = stats_df(df["time_since_origin_ts"], "Time Since Origin (s)") if "time_since_origin_ts" in df.columns else pd.DataFrame()

if "hop_number" in df.columns and "label" in df.columns:
    hop_vals = sorted(df["hop_number"].dropna().unique())
    hop_fraud = pd.DataFrame({
        "Hop Number": hop_vals,
        "Txn Count":  [(df["hop_number"]==h).sum() for h in hop_vals],
        "Fraud Count":[int(df[df["hop_number"]==h]["label"].sum()) for h in hop_vals],
        "Fraud Rate (%)":[round(df[df["hop_number"]==h]["label"].mean()*100, 2) for h in hop_vals],
    })
else:
    hop_fraud = pd.DataFrame()

print("✅ Section 8 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — SHARED IDENTITY SIGNALS
# ══════════════════════════════════════════════════════════════════════════════
shared_rows = []
for col, txt in [("shared_kyc_id","Shared KYC ID"),("shared_phone_hash","Shared Phone Hash"),
                  ("shared_email_hash","Shared Email Hash")]:
    if col in df.columns:
        has_shared = df[col].notna() & (df[col].astype(str).str.strip().isin(["","nan"]) == False)
        fraud_rate = round(df.loc[has_shared,"label"].mean()*100,2) if "label" in df.columns and has_shared.sum()>0 else None
        shared_rows.append({
            "Signal": txt, "Txns with Shared Value": int(has_shared.sum()),
            "Percentage (%)": round(has_shared.mean()*100, 2),
            "Unique Shared Values": df.loc[has_shared, col].nunique(),
            "Fraud Rate if Shared (%)": fraud_rate,
        })
shared_identity_df = pd.DataFrame(shared_rows) if shared_rows else pd.DataFrame()
print("✅ Section 9 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTIONS 10–14 — VELOCITY & BALANCE FEATURES
# ══════════════════════════════════════════════════════════════════════════════

# Sender account velocity
sender_acct_cols = [c for c in df.columns if c.startswith("sender_acct_")]
sender_acct_stats  = velocity_stats_table(sender_acct_cols)
sender_acct_flcomp = fraud_vs_legit_stats(sender_acct_cols)

# Sender customer velocity
sender_cust_cols = [c for c in df.columns if c.startswith("sender_cust_") and c != "sender_cust_id_for_rollup"]
sender_cust_stats  = velocity_stats_table(sender_cust_cols)
sender_cust_flcomp = fraud_vs_legit_stats(sender_cust_cols)

# Sender balance
sender_bal_cols = [c for c in df.columns if c.startswith("sender_bal") or c.startswith("sender_running") or c.startswith("sender_balance") or c.startswith("sender_cumul") or c == "sender_current_balance"]
sender_bal_stats  = velocity_stats_table(sender_bal_cols)
sender_bal_flcomp = fraud_vs_legit_stats(sender_bal_cols)

if "sender_balance_after_txn" in df.columns:
    neg_bal = df[df["sender_balance_after_txn"] < 0]
    neg_bal_summary = pd.DataFrame({
        "Metric": ["Txns with Negative Balance After", "Percentage (%)", "Min Balance After"],
        "Value":  [len(neg_bal), round(len(neg_bal)/TOTAL*100, 2), round(df["sender_balance_after_txn"].min(), 2)],
    })
else:
    neg_bal_summary = pd.DataFrame()

# Receiver account velocity
receiver_acct_cols = [c for c in df.columns if c.startswith("receiver_acct_")]
receiver_acct_stats  = velocity_stats_table(receiver_acct_cols)
receiver_acct_flcomp = fraud_vs_legit_stats(receiver_acct_cols)

# NaN rate for receiver (external bene = NaN)
if receiver_acct_cols:
    recv_null_df = pd.DataFrame({
        "Feature": receiver_acct_cols,
        "Null Rate (%)": [round(df[c].isnull().mean()*100, 2) for c in receiver_acct_cols],
        "Note": ["NaN = external beneficiary"]*len(receiver_acct_cols),
    })
else:
    recv_null_df = pd.DataFrame()

# Receiver balance
receiver_bal_cols = [c for c in df.columns if c.startswith("receiver_bal") or c.startswith("receiver_running") or c.startswith("receiver_balance") or c.startswith("receiver_cumul") or c == "receiver_current_balance"]
receiver_bal_stats  = velocity_stats_table(receiver_bal_cols)
receiver_bal_flcomp = fraud_vs_legit_stats(receiver_bal_cols)

print(f"✅ Sections 10–14 done")
print(f"   Sender acct cols: {len(sender_acct_cols)} | Sender cust cols: {len(sender_cust_cols)}")
print(f"   Sender bal cols: {len(sender_bal_cols)} | Receiver acct cols: {len(receiver_acct_cols)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTIONS 15–16 — RULE ENGINE & LIFT ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
rule_flag_cols = [c for c in df.columns if c.startswith("rule_") and c not in
                  ["rule_trigger_count","max_rule_severity","weighted_rule_score"]]
rules_df      = rule_summary_fn(rule_flag_cols)
trigger_stats = stats_df(df["rule_trigger_count"], "Rule Trigger Count") if "rule_trigger_count" in df.columns else pd.DataFrame()
severity_df   = pct_df(df["max_rule_severity"], "Max Rule Severity") if "max_rule_severity" in df.columns else pd.DataFrame()
wscore_stats  = stats_df(df["weighted_rule_score"], "Weighted Rule Score") if "weighted_rule_score" in df.columns else pd.DataFrame()
trig_dist     = pct_df(df["rule_trigger_count"].astype(int), "Trigger Count") if "rule_trigger_count" in df.columns else pd.DataFrame()

if "label" in df.columns and rule_flag_cols:
    fraud_r = df[df["label"]==1][rule_flag_cols].mean().round(4)
    legit_r = df[df["label"]==0][rule_flag_cols].mean().round(4)
    rule_compare = pd.DataFrame({
        "Rule": [c.replace("rule_","").replace("_"," ").title() for c in rule_flag_cols],
        "Column": rule_flag_cols,
        "Fraud Fire (%)": (fraud_r.values*100).round(2),
        "Legit Fire (%)": (legit_r.values*100).round(2),
        "Lift (F/L)": (fraud_r.values / (legit_r.values + 1e-9)).round(2),
    }).sort_values("Lift (F/L)", ascending=False).reset_index(drop=True)
else:
    rule_compare = pd.DataFrame()

print(f"✅ Sections 15–16 done | Rules found: {len(rule_flag_cols)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 17 — FRAUD INTENSITY SCORE (FIS)
# ══════════════════════════════════════════════════════════════════════════════
fis_stats     = stats_df(df["fraud_intensity_score"], "FIS (0–100)") if "fraud_intensity_score" in df.columns else pd.DataFrame()
fis_raw_stats = stats_df(df["fraud_intensity_score_raw"], "FIS Raw") if "fraud_intensity_score_raw" in df.columns else pd.DataFrame()
fis_band_df   = pct_df(df["fis_band"], "FIS Band") if "fis_band" in df.columns else pd.DataFrame()
fis_band_fraud= fraud_legit_compare(df["fis_band"], "FIS Band") if "fis_band" in df.columns and "label" in df.columns else pd.DataFrame()

if "fraud_intensity_score" in df.columns and "label" in df.columns:
    fis_by_label = pd.DataFrame({
        "Group": ["Fraud (label=1)", "Legitimate (label=0)", "All Transactions"],
        "Mean FIS": [round(df[df["label"]==1]["fraud_intensity_score"].mean(),2),
                     round(df[df["label"]==0]["fraud_intensity_score"].mean(),2),
                     round(df["fraud_intensity_score"].mean(),2)],
        "Median FIS": [round(df[df["label"]==1]["fraud_intensity_score"].median(),2),
                       round(df[df["label"]==0]["fraud_intensity_score"].median(),2),
                       round(df["fraud_intensity_score"].median(),2)],
        "Max FIS": [round(df[df["label"]==1]["fraud_intensity_score"].max(),2),
                    round(df[df["label"]==0]["fraud_intensity_score"].max(),2),
                    round(df["fraud_intensity_score"].max(),2)],
        "Std FIS": [round(df[df["label"]==1]["fraud_intensity_score"].std(),2),
                    round(df[df["label"]==0]["fraud_intensity_score"].std(),2),
                    round(df["fraud_intensity_score"].std(),2)],
    })
else:
    fis_by_label = pd.DataFrame()

print("✅ Section 17 done")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTIONS 18–19 — FRAUD EXAMPLES & WIDE TABLE SETUP
# ══════════════════════════════════════════════════════════════════════════════

FRAUD_TYPE_DESCRIPTIONS = {
    "mule_ring": ("Mule Ring",
        "A network of accounts used to receive and rapidly forward illicit funds. "
        "Each mule account receives deposits and immediately transfers funds onward, "
        "fragmenting the money trail across multiple hops."),
    "layering": ("Layering",
        "Funds are moved through a rapid chain of transactions to obscure their origin. "
        "Each hop is slightly smaller (decay pattern) and executed within minutes, "
        "making it hard to trace back to the original illicit source."),
    "ATO": ("Account Takeover (ATO)",
        "A fraudster hijacks a legitimate account via credential stuffing or phishing, "
        "then drains funds using unusual channels, devices, or times inconsistent with "
        "the account owner's normal behaviour."),
    "smurfing": ("Smurfing (Structuring)",
        "Cash is broken into multiple smaller transactions, each deliberately kept "
        "below the ₹10,000 regulatory reporting threshold to avoid CTR filing."),
    "identity_fraud": ("Identity Fraud",
        "A fraudulent account opened using stolen or synthetic identity documents, "
        "typically new (<60 days old), immediately used for high-value transactions."),
    "dormant_ATO": ("Dormant Account Takeover",
        "A dormant (inactive >30 days) account taken over using a new unrecognised "
        "device. Reactivation burst with large amounts to high-risk beneficiaries."),
    "dormant_smurfing": ("Dormant Account Smurfing",
        "A reactivated dormant account used for cash structuring: multiple cash "
        "transactions just below ₹10,000 within days of reactivation."),
    "dormant_to_offshore": ("Dormant Account — Offshore Wire",
        "Funds parked in a dormant account suddenly wired offshore or to crypto "
        "immediately upon reactivation — the integration step of a laundering scheme."),
}

def _val(row, col, default="N/A"):
    v = row.get(col, default)
    return default if (v is None or (isinstance(v, float) and np.isnan(v))) else v

def build_fraud_reasoning(row, fraud_type):
    signals = []
    amt = _val(row, "amount", 0)
    if fraud_type == "mule_ring":
        signals.append(f"High-value transfer ₹{amt:,.2f} through a mule account")
        v24 = _val(row, "sender_acct_txn_count_24h", 0)
        if str(v24) != "N/A" and float(v24) > 10: signals.append(f"Account processed {v24} txns in 24h — rapid forwarding")
        if _val(row,"beneficiary_type") in ("crypto","offshore"): signals.append(f"Funds exit to {_val(row,'beneficiary_type')} — high-risk exit node")
        if _val(row,"is_night",0)==1: signals.append("Transaction at night — outside normal banking hours")
        if _val(row,"customer_risk_rating")=="very_high": signals.append("Customer flagged as very-high risk")
    elif fraud_type == "layering":
        signals.append(f"₹{amt:,.2f} is part of a rapid multi-hop chain")
        v1 = _val(row, "sender_acct_txn_count_1h", 0)
        if str(v1) != "N/A" and float(v1) > 2: signals.append(f"{v1} txns in last 1h — rapid hop sequence")
        if _val(row,"beneficiary_type") in ("crypto","offshore"): signals.append(f"Final hop exits to {_val(row,'beneficiary_type')}")
        fid = _val(row,"synthetic_flow_id","N/A")
        if fid != "N/A": signals.append(f"Flow ID: {fid} | Hop {_val(row,'hop_number')} / Depth {_val(row,'flow_depth')}")
    elif fraud_type == "ATO":
        signals.append(f"Large withdrawal ₹{amt:,.2f} on potentially hijacked account")
        d = _val(row,"device_age_days",9999)
        if str(d) != "N/A" and float(d) < 90: signals.append(f"Device only {d} days old — unrecognised")
        if _val(row,"rooted_flag",0)==1: signals.append("Rooted/jailbroken device — malware risk")
        if _val(row,"vpn_flag",0)==1: signals.append("VPN active — attacker masking location")
        ip = _val(row,"ip_risk_score",0)
        if str(ip) != "N/A" and float(ip) > 0.6: signals.append(f"IP risk score {ip:.2f} — suspicious IP")
    elif fraud_type == "smurfing":
        signals.append(f"Cash ₹{amt:,.2f} just below ₹10,000 CTR threshold")
        if _val(row,"cash_flag",0)==1: signals.append("Cash flag = 1 — physical cash involved")
        v7 = _val(row,"sender_acct_txn_count_7d",0)
        if str(v7) != "N/A" and float(v7) > 3: signals.append(f"{v7} txns in 7d — repeated sub-threshold pattern")
    elif fraud_type == "identity_fraud":
        signals.append(f"High-value ₹{amt:,.2f} on newly opened account")
        a = _val(row,"account_open_days",9999)
        if str(a) != "N/A" and float(a) < 60: signals.append(f"Account only {a} days old — synthetic identity")
        if _val(row,"kyc_level")=="low": signals.append("KYC level = low — minimal identity verification")
    elif fraud_type == "dormant_ATO":
        signals.append(f"Dormant account reactivated with ₹{amt:,.2f}")
        if _val(row,"dormancy_flag",0)==1: signals.append("dormancy_flag = 1 — confirms reactivation after long silence")
        d = _val(row,"device_age_days",9999)
        if str(d) != "N/A" and float(d) < 90: signals.append(f"New device {d} days old — unrecognised")
        if _val(row,"beneficiary_type") in ("crypto","offshore"): signals.append(f"Immediate transfer to {_val(row,'beneficiary_type')}")
    elif fraud_type == "dormant_smurfing":
        signals.append(f"Dormant account reactivated for structuring (₹{amt:,.2f}/txn)")
        if _val(row,"dormancy_flag",0)==1: signals.append("dormancy_flag = 1 — follows >30-day silence")
        if _val(row,"cash_flag",0)==1: signals.append("Cash flag = 1 — physical cash structuring")
    elif fraud_type == "dormant_to_offshore":
        signals.append(f"Dormant account offshore wire ₹{amt:,.2f}")
        if _val(row,"dormancy_flag",0)==1: signals.append("dormancy_flag = 1 — account silent >30d")
        if _val(row,"beneficiary_type") in ("crypto","offshore"): signals.append(f"Beneficiary = {_val(row,'beneficiary_type')} — integration step")
        if _val(row,"beneficiary_country_risk")=="high": signals.append("Destination is high-risk jurisdiction")
    else:
        signals.append(f"₹{amt:,.2f} flagged as {fraud_type}")
    return signals

# Build examples
EXAMPLE_COLS = [c for c in [
    "transaction_id","sender_account_id","receiver_account_id","customer_id",
    "timestamp","amount","channel","transaction_type","cash_flag",
    "dormancy_flag","is_night","is_weekend","is_early_morning",
    "beneficiary_type","beneficiary_country_risk",
    "kyc_level","country_risk","occupation","income_bracket",
    "customer_risk_rating","pep_flag","device_age_days",
    "rooted_flag","vpn_flag","emulator_flag","ip_risk_score",
    "account_open_days","avg_balance",
    "synthetic_flow_id","flow_depth","hop_number",
    "sender_acct_txn_count_1h","sender_acct_txn_count_24h","sender_acct_txn_count_7d",
    "sender_balance_before_txn","sender_balance_after_txn","sender_bal_ratio_after_to_current",
    "rule_trigger_count","max_rule_severity","weighted_rule_score",
    "fraud_intensity_score","fis_band","fraud_type","label",
] if c in df.columns]

fraud_examples = {}
if "fraud_type" in df.columns and "label" in df.columns:
    for ftype in df[df["label"]==1]["fraud_type"].dropna().unique():
        subset = df[(df["fraud_type"]==ftype) & (df["label"]==1)]
        if subset.empty: continue
        sort_cols = [c for c in ["fraud_intensity_score","weighted_rule_score","amount"] if c in subset.columns]
        ex_row = subset.sort_values(sort_cols, ascending=False).iloc[0]
        ex_df  = ex_row[EXAMPLE_COLS].reset_index(drop=True).to_frame("Value")
        ex_df.index.name = "Field"
        ex_df  = ex_df.reset_index()
        ex_df["Value"] = ex_df["Value"].astype(str)
        fraud_examples[ftype] = (ex_df, build_fraud_reasoning(ex_row, ftype))

print(f"✅ Sections 18–19 data ready | Fraud types: {list(fraud_examples.keys())}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXCEL WRITE HELPERS
# ══════════════════════════════════════════════════════════════════════════════

TYPE_COLOURS      = ["C00000","7030A0","0070C0","FF8C00","00B050","C9511F","833C00","4472C4"]
TYPE_HDR2_COLOURS = ["FFD7D7","E8D5F5","D0E8F7","FFE8C8","D5F0DC","F5D7C0","EAD6C0","D0DCF5"]
ALT_COLOURS       = ["FFF0F0","F7F0FF","EEF6FF","FFF5E8","EEFAF2","FDF3EC","F7EDE3","EEF2FF"]

def write_fraud_example_block(ws, sr, sc, ftype, example_df, signals, descriptions):
    display_name, description = descriptions.get(ftype, (ftype.replace("_"," ").title(), ""))
    ec = sc + 1
    ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
    c = ws.cell(row=sr, column=sc, value=f"🔴  {display_name}")
    c.font = Font(name="Arial", bold=True, size=12, color="FFFFFF")
    c.fill = PatternFill("solid", start_color="C00000")
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = THIN; ws.row_dimensions[sr].height = 24; sr += 1
    ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
    c = ws.cell(row=sr, column=sc, value=description)
    c.font = Font(name="Arial", italic=True, size=9, color="1F2D3D")
    c.fill = WARN_FILL; c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    c.border = THIN; ws.row_dimensions[sr].height = 42; sr += 1
    ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
    c = ws.cell(row=sr, column=sc, value="⚠️  Why flagged?")
    c.font = Font(name="Arial", bold=True, size=10, color="FFFFFF")
    c.fill = PatternFill("solid", start_color="F4B942")
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = THIN; ws.row_dimensions[sr].height = 20; sr += 1
    for i, signal in enumerate(signals):
        ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
        c = ws.cell(row=sr, column=sc, value=f"  ✦ {signal}")
        c.font = Font(name="Arial", size=9, color="1F2D3D")
        c.fill = ALT_FILL if i%2==1 else WHITE
        c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        c.border = THIN; ws.row_dimensions[sr].height = 16; sr += 1
    ws.merge_cells(start_row=sr, start_column=sc, end_row=sr, end_column=ec)
    c = ws.cell(row=sr, column=sc, value="📋  Representative Transaction (highest FIS in typology)")
    c.font = Font(name="Arial", bold=True, size=10, color="1F2D3D")
    c.fill = PatternFill("solid", start_color="E8D5F5")
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = THIN; ws.row_dimensions[sr].height = 20; sr += 1
    hdr(ws, sr, sc, "Field", size=9, fill=HDR2_FILL, color="1F2D3D")
    hdr(ws, sr, sc+1, "Value", size=9, fill=HDR2_FILL, color="1F2D3D"); sr += 1
    for ri, row_data in example_df.iterrows():
        alt = ri%2==1
        dat(ws, sr, sc, row_data["Field"], alt=alt)
        dat(ws, sr, sc+1, row_data["Value"], alt=alt)
        ws.row_dimensions[sr].height = 15; sr += 1
    ws.column_dimensions[get_column_letter(sc)].width   = 34
    ws.column_dimensions[get_column_letter(sc+1)].width = 50
    return sr + 2

def banner(ws, row, text, n_cols, hex_color, height=26):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n_cols)
    c = ws.cell(row=row, column=1, value=text)
    c.font = Font(name="Arial", bold=True, size=12, color="FFFFFF")
    c.fill = PatternFill("solid", start_color=hex_color)
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border = THIN; ws.row_dimensions[row].height = height
    return row+1

def info_row(ws, row, text, n_cols, bg_hex, italic=False, height=32):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n_cols)
    c = ws.cell(row=row, column=1, value=text)
    c.font = Font(name="Arial", italic=italic, size=9, color="1F2D3D")
    c.fill = PatternFill("solid", start_color=bg_hex)
    c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    c.border = THIN; ws.row_dimensions[row].height = height
    return row+1

print("✅ Excel write helpers ready")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD EXCEL WORKBOOK — All 19 sheets
# ══════════════════════════════════════════════════════════════════════════════

wb = Workbook()

# ─── SHEET 1: Dashboard ───────────────────────────────────────────────────────
ws0 = wb.active; ws0.title = "📊 Dashboard"; ws0.sheet_view.showGridLines = False
ws0.merge_cells("A1:F1")
hdr(ws0, 1, 1, "SmartSentry AML — Transaction Data EDA Report (V3)", size=15, fill=HDR_FILL)
ws0.row_dimensions[1].height = 40
ws0.merge_cells("A2:F2")
hdr(ws0, 2, 1,
    f"Total: {TOTAL:,} txns  |  Fraud rate: {fraud_pct}%  |  Customers: {df['customer_id'].nunique() if 'customer_id' in df.columns else 'N/A'}  |  Columns: {df.shape[1]}",
    size=10, fill=SUB_FILL)
r = 4
r = write_block(ws0, r, 1, "📌 Key Performance Indicators", overview_kpis)
if not fraud_type_df.empty:  r = write_block(ws0, r, 1, "🎭 Fraud Type Breakdown", fraud_type_df)
if not null_audit.empty:     r = write_block(ws0, r, 1, "🔍 Null Audit (columns with missing values)", null_audit.head(30))
print("✅ Sheet 1: Dashboard")

# ─── SHEET 2: Transactions ────────────────────────────────────────────────────
ws1 = wb.create_sheet("💳 Transactions"); ws1.sheet_view.showGridLines = False
ws1.merge_cells("A1:J1"); hdr(ws1,1,1,"TRANSACTION CHARACTERISTICS",size=13,fill=HDR_FILL); ws1.row_dimensions[1].height=35
r=3
r=write_block(ws1,r,1,"Amount Statistics (INR)",amount_stats)
r=write_block(ws1,r,1,"Channel Distribution",channel_df)
r=write_block(ws1,r,1,"Transaction Type",txn_type_df)
r=write_block(ws1,r,1,"Channel × Fraud Rate",channel_fraud)
r=write_block(ws1,r,1,"Transaction Type × Fraud Rate",txn_type_fraud)
r2=3
r2=write_block(ws1,r2,5,"Debit / Credit Split",debit_df)
r2=write_block(ws1,r2,5,"Cash Flag",cash_df)
r2=write_block(ws1,r2,5,"Dormancy Flag (Transactions)",dormancy_df)
r2=write_block(ws1,r2,5,"Dormancy by Account",dormant_acct_df)
print("✅ Sheet 2: Transactions")

# ─── SHEET 3: Temporal ────────────────────────────────────────────────────────
ws2 = wb.create_sheet("⏰ Temporal"); ws2.sheet_view.showGridLines = False
ws2.merge_cells("A1:K1"); hdr(ws2,1,1,"TEMPORAL DISTRIBUTION",size=13,fill=HDR_FILL); ws2.row_dimensions[1].height=35
r=3
r=write_block(ws2,r,1,"Hour of Day",hour_df)
r=write_block(ws2,r,1,"Day of Week",dow_df)
r=write_block(ws2,r,1,"Month",month_df)
r=write_block(ws2,r,1,"Quarter",quarter_df)
r=write_block(ws2,r,1,"Day of Month",dom_df.head(15))
r2=3
r2=write_block(ws2,r2,5,"Night Transactions (22–05h)",night_df)
r2=write_block(ws2,r2,5,"Weekend Transactions",weekend_df)
r2=write_block(ws2,r2,5,"Business Hours (09–17h M–F)",biz_df)
r2=write_block(ws2,r2,5,"Early Morning (05–09h)",early_df)
r2=write_block(ws2,r2,5,"Fraud Rate by Time Window",temporal_fraud_df)
print("✅ Sheet 3: Temporal")

# ─── SHEET 4: Customer & Account ─────────────────────────────────────────────
ws3 = wb.create_sheet("👤 Customer & Account"); ws3.sheet_view.showGridLines = False
ws3.merge_cells("A1:L1"); hdr(ws3,1,1,"CUSTOMER & ACCOUNT PROFILE",size=13,fill=HDR_FILL); ws3.row_dimensions[1].height=35
r=3
r=write_block(ws3,r,1,"Occupation",occ_df)
r=write_block(ws3,r,1,"Industry",industry_df)
r=write_block(ws3,r,1,"Account Type",acc_type_df)
r=write_block(ws3,r,1,"Avg Balance (INR) Stats",avg_bal_stats)
r=write_block(ws3,r,1,"Account Age (Days)",acct_age_stats)
r2=3
r2=write_block(ws3,r2,5,"KYC Level",kyc_df)
r2=write_block(ws3,r2,5,"Country Risk",crisk_df)
r2=write_block(ws3,r2,5,"Customer Risk Rating",cust_risk_df)
r2=write_block(ws3,r2,5,"Income Bracket",inc_df)
r2=write_block(ws3,r2,5,"PEP Flag",pep_df)
r2=write_block(ws3,r2,5,"Customer Risk Rating (Num)",crr_num_stats)
r2=write_block(ws3,r2,5,"KYC Level (Num)",kyc_num_stats)
r3=3
r3=write_block(ws3,r3,9,"KYC Level × Fraud Rate",kyc_fraud)
r3=write_block(ws3,r3,9,"Country Risk × Fraud Rate",crisk_fraud)
r3=write_block(ws3,r3,9,"Income Bracket × Fraud Rate",inc_fraud)
r3=write_block(ws3,r3,9,"Customer Risk Rating × Fraud Rate",crr_fraud)
print("✅ Sheet 4: Customer & Account")

# ─── SHEET 5: Devices & Network ───────────────────────────────────────────────
ws4 = wb.create_sheet("📱 Devices & Network"); ws4.sheet_view.showGridLines = False
ws4.merge_cells("A1:J1"); hdr(ws4,1,1,"DEVICE & NETWORK SIGNALS",size=13,fill=HDR_FILL); ws4.row_dimensions[1].height=35
r=3
r=write_block(ws4,r,1,"OS Type",os_df)
r=write_block(ws4,r,1,"Rooted Device",rooted_df,flag_high=3.0)
r=write_block(ws4,r,1,"VPN / Proxy Active",vpn_df,flag_high=5.0)
r=write_block(ws4,r,1,"Emulator Detected",emu_df,flag_high=2.0)
r=write_block(ws4,r,1,"Device Age (Days) Stats",dev_age_stats)
r2=3
r2=write_block(ws4,r2,5,"Device Signal × Fraud Rate",device_fraud_df)
print("✅ Sheet 5: Devices & Network")

# ─── SHEET 6: Beneficiaries ───────────────────────────────────────────────────
ws5 = wb.create_sheet("💸 Beneficiaries"); ws5.sheet_view.showGridLines = False
ws5.merge_cells("A1:J1"); hdr(ws5,1,1,"BENEFICIARY PROFILE",size=13,fill=HDR_FILL); ws5.row_dimensions[1].height=35
r=3
r=write_block(ws5,r,1,"Beneficiary Type",ben_type_df)
r=write_block(ws5,r,1,"Beneficiary Country Risk",ben_risk_df,flag_high=5.0)
r2=3
r2=write_block(ws5,r2,5,"Beneficiary Type × Fraud Rate",ben_fraud)
r2=write_block(ws5,r2,5,"Beneficiary Country Risk × Fraud Rate",benrisk_fraud)
print("✅ Sheet 6: Beneficiaries")

# ─── SHEET 7: IP & Geo ────────────────────────────────────────────────────────
ws6 = wb.create_sheet("🌐 IP & Geo"); ws6.sheet_view.showGridLines = False
ws6.merge_cells("A1:J1"); hdr(ws6,1,1,"IP & GEOLOCATION FEATURES",size=13,fill=HDR_FILL); ws6.row_dimensions[1].height=35
r=3
r=write_block(ws6,r,1,"IP Risk Score Statistics",ip_risk_stats)
r=write_block(ws6,r,1,"IP Risk Score Bands",ip_bucket_df)
r=write_block(ws6,r,1,"IP Risk Band × Fraud Rate",ip_fraud_df)
r=write_block(ws6,r,1,"Geo Latitude Statistics",geo_lat_stats)
r=write_block(ws6,r,1,"Geo Longitude Statistics",geo_lon_stats)
r2=3
r2=write_block(ws6,r2,5,"Top 20 Home Cities",home_city_df)
print("✅ Sheet 7: IP & Geo")

# ─── SHEET 8: Flow & Graph ────────────────────────────────────────────────────
ws7 = wb.create_sheet("🔗 Flow & Graph"); ws7.sheet_view.showGridLines = False
ws7.merge_cells("A1:J1"); hdr(ws7,1,1,"FLOW / GRAPH FEATURES  (mule_ring & layering typologies)",size=13,fill=HDR_FILL); ws7.row_dimensions[1].height=35
r=3
r=write_block(ws7,r,1,"Flow ID Coverage",flow_summary)
r=write_block(ws7,r,1,"Flow Depth Statistics",flow_depth_stats)
r=write_block(ws7,r,1,"Hop Number Statistics",hop_stats)
r=write_block(ws7,r,1,"Time Since Origin (s) Statistics",time_origin_stats)
r2=3
r2=write_block(ws7,r2,5,"Fraud Rate by Hop Number",hop_fraud)
print("✅ Sheet 8: Flow & Graph")

# ─── SHEET 9: Shared Identity ─────────────────────────────────────────────────
ws8 = wb.create_sheet("🪪 Shared Identity"); ws8.sheet_view.showGridLines = False
ws8.merge_cells("A1:J1"); hdr(ws8,1,1,"SHARED IDENTITY SIGNALS",size=13,fill=HDR_FILL); ws8.row_dimensions[1].height=35
r=3
r=write_block(ws8,r,1,"Shared Identity Signal Summary",shared_identity_df)
print("✅ Sheet 9: Shared Identity")

# ─── SHEET 10: Sender Acct Velocity ──────────────────────────────────────────
ws9 = wb.create_sheet("⚡ Sender Acct Velocity"); ws9.sheet_view.showGridLines = False
ws9.merge_cells("A1:L1"); hdr(ws9,1,1,"SENDER ACCOUNT VELOCITY  (1h/24h/7d/30d)",size=13,fill=HDR_FILL); ws9.row_dimensions[1].height=35
r=3
r=write_block(ws9,r,1,"Describe Statistics (all windows)",sender_acct_stats)
r=write_block(ws9,r,1,"Fraud vs Legit Mean Comparison",sender_acct_flcomp)
print("✅ Sheet 10: Sender Acct Velocity")

# ─── SHEET 11: Sender Cust Velocity ──────────────────────────────────────────
ws10 = wb.create_sheet("👥 Sender Cust Velocity"); ws10.sheet_view.showGridLines = False
ws10.merge_cells("A1:L1"); hdr(ws10,1,1,"SENDER CUSTOMER VELOCITY  (cross-account, 1h/24h/7d/30d)",size=13,fill=HDR_FILL); ws10.row_dimensions[1].height=35
r=3
r=write_block(ws10,r,1,"Describe Statistics (all windows)",sender_cust_stats)
r=write_block(ws10,r,1,"Fraud vs Legit Mean Comparison",sender_cust_flcomp)
print("✅ Sheet 11: Sender Cust Velocity")

# ─── SHEET 12: Sender Balance ─────────────────────────────────────────────────
ws11 = wb.create_sheet("💰 Sender Balance"); ws11.sheet_view.showGridLines = False
ws11.merge_cells("A1:L1"); hdr(ws11,1,1,"SENDER BALANCE FEATURES",size=13,fill=HDR_FILL); ws11.row_dimensions[1].height=35
r=3
r=write_block(ws11,r,1,"Balance Feature Statistics",sender_bal_stats)
r=write_block(ws11,r,1,"Fraud vs Legit Mean Comparison",sender_bal_flcomp)
r=write_block(ws11,r,1,"Negative Balance Audit",neg_bal_summary)
print("✅ Sheet 12: Sender Balance")

# ─── SHEET 13: Receiver Acct Velocity ────────────────────────────────────────
ws12 = wb.create_sheet("📥 Receiver Acct Velocity"); ws12.sheet_view.showGridLines = False
ws12.merge_cells("A1:L1"); hdr(ws12,1,1,"RECEIVER ACCOUNT VELOCITY  (NaN = external beneficiary)",size=13,fill=HDR_FILL); ws12.row_dimensions[1].height=35
r=3
r=write_block(ws12,r,1,"Describe Statistics (all windows)",receiver_acct_stats)
r=write_block(ws12,r,1,"Fraud vs Legit Mean Comparison",receiver_acct_flcomp)
r=write_block(ws12,r,1,"NaN Rate per Feature",recv_null_df)
print("✅ Sheet 13: Receiver Acct Velocity")

# ─── SHEET 14: Receiver Balance ───────────────────────────────────────────────
ws13 = wb.create_sheet("📤 Receiver Balance"); ws13.sheet_view.showGridLines = False
ws13.merge_cells("A1:L1"); hdr(ws13,1,1,"RECEIVER BALANCE FEATURES",size=13,fill=HDR_FILL); ws13.row_dimensions[1].height=35
r=3
r=write_block(ws13,r,1,"Balance Feature Statistics",receiver_bal_stats)
r=write_block(ws13,r,1,"Fraud vs Legit Mean Comparison",receiver_bal_flcomp)
print("✅ Sheet 14: Receiver Balance")

# ─── SHEET 15: Rule Engine ────────────────────────────────────────────────────
ws14 = wb.create_sheet("🔴 Rule Engine"); ws14.sheet_view.showGridLines = False
ws14.merge_cells("A1:J1"); hdr(ws14,1,1,"AML RULE ENGINE — TRIGGER ANALYSIS",size=13,fill=HDR_FILL); ws14.row_dimensions[1].height=35
r=3
r=write_block(ws14,r,1,f"All {len(rule_flag_cols)} Rules — Trigger Frequency",rules_df)
r2=3
r2=write_block(ws14,r2,8,"Rule Trigger Count Stats",trigger_stats)
r2=write_block(ws14,r2,8,"Trigger Count Distribution (top 20)",trig_dist.head(20))
r2=write_block(ws14,r2,8,"Max Severity Distribution",severity_df)
r2=write_block(ws14,r2,8,"Weighted Rule Score Stats",wscore_stats)
print("✅ Sheet 15: Rule Engine")

# ─── SHEET 16: Rule Lift Analysis ────────────────────────────────────────────
if not rule_compare.empty:
    ws15 = wb.create_sheet("📈 Rule Lift Analysis"); ws15.sheet_view.showGridLines = False
    ws15.merge_cells("A1:F1"); hdr(ws15,1,1,"RULE LIFT ANALYSIS — Fraud vs Legitimate Fire Rates",size=13,fill=HDR_FILL)
    ws15.merge_cells("A2:F2"); hdr(ws15,2,1,"Lift > 1.0 = rule fires more on fraud. Higher = stronger discriminating power.",size=9,fill=SUB_FILL)
    ws15.row_dimensions[1].height=35; ws15.row_dimensions[2].height=22
    write_block(ws15,4,1,"Rule Discrimination Power (sorted by Lift desc)",rule_compare)
    print("✅ Sheet 16: Rule Lift Analysis")

# ─── SHEET 17: FIS Scoring ────────────────────────────────────────────────────
ws16 = wb.create_sheet("🎯 FIS Scoring"); ws16.sheet_view.showGridLines = False
ws16.merge_cells("A1:J1"); hdr(ws16,1,1,"FRAUD INTENSITY SCORE (FIS) ANALYSIS",size=13,fill=HDR_FILL)
ws16.merge_cells("A2:J2")
hdr(ws16,2,1,"FIS = clip(rule_trigger_count×2.5, 60) + label×25 + clip(amt_to_bal_ratio, 0–10)/10×15  |  Final clip [0,100]",size=9,fill=SUB_FILL)
ws16.row_dimensions[1].height=35; ws16.row_dimensions[2].height=24
r=4
r=write_block(ws16,r,1,"FIS (0–100) Statistics",fis_stats)
r=write_block(ws16,r,1,"FIS Raw Statistics",fis_raw_stats)
r=write_block(ws16,r,1,"FIS Band Distribution",fis_band_df)
r2=4
r2=write_block(ws16,r2,5,"FIS by Label (Fraud vs Legit)",fis_by_label)
r2=write_block(ws16,r2,5,"FIS Band × Fraud Rate",fis_band_fraud)
print("✅ Sheet 17: FIS Scoring")

# ─── SHEET 18: Fraud Examples ─────────────────────────────────────────────────
if fraud_examples:
    ws17 = wb.create_sheet("🔍 Fraud Examples"); ws17.sheet_view.showGridLines = False
    ws17.merge_cells("A1:B1"); hdr(ws17,1,1,"FRAUD TYPOLOGY EXAMPLES — Highest-Risk Transaction per Type",size=13,fill=HDR_FILL); ws17.row_dimensions[1].height=35
    ws17.merge_cells("A2:B2"); hdr(ws17,2,1,"Ranked by fraud_intensity_score. Each block: description → signals → representative transaction.",size=9,fill=SUB_FILL); ws17.row_dimensions[2].height=24
    ordered_keys = [k for k in ["mule_ring","layering","ATO","smurfing","identity_fraud",
                                  "dormant_ATO","dormant_smurfing","dormant_to_offshore"] if k in fraud_examples]
    ordered_keys += [k for k in fraud_examples if k not in ordered_keys]
    r_left, r_right = 4, 4
    for i, ftype in enumerate(ordered_keys):
        ex_df, sigs = fraud_examples[ftype]
        if i%2==0: r_left  = write_fraud_example_block(ws17, r_left,  1, ftype, ex_df, sigs, FRAUD_TYPE_DESCRIPTIONS)
        else:      r_right = write_fraud_example_block(ws17, r_right, 4, ftype, ex_df, sigs, FRAUD_TYPE_DESCRIPTIONS)
    print("✅ Sheet 18: Fraud Examples")

# ─── SHEET 19: Fraud Txn Wide Table ──────────────────────────────────────────
WIDE_COLS = [c for c in [
    "transaction_id","sender_account_id","receiver_account_id","device_id","customer_id",
    "timestamp","amount","channel","transaction_type","debit_credit",
    "cash_flag","dormancy_flag","is_night","is_weekend","is_early_morning",
    "beneficiary_id","beneficiary_type","beneficiary_country_risk",
    "kyc_level","country_risk","occupation","income_bracket",
    "customer_risk_rating","pep_flag","account_type",
    "device_age_days","rooted_flag","vpn_flag","emulator_flag","ip_risk_score",
    "account_open_days","avg_balance",
    "synthetic_flow_id","flow_depth","hop_number","time_since_origin_ts",
    "shared_kyc_id","shared_phone_hash","shared_email_hash",
    "sender_acct_txn_count_1h","sender_acct_txn_count_24h","sender_acct_txn_count_7d",
    "sender_acct_outflow_amt_24h","sender_acct_outflow_amt_7d",
    "sender_cust_txn_count_24h","sender_cust_outflow_amt_24h",
    "sender_balance_before_txn","sender_balance_after_txn","sender_bal_ratio_after_to_current",
    "receiver_acct_txn_count_24h","receiver_acct_inflow_amt_24h","receiver_balance_after_txn",
    "rule_trigger_count","max_rule_severity","weighted_rule_score",
    "fraud_intensity_score","fis_band","fraud_type","label",
] if c in df.columns]

if "fraud_type" in df.columns and "label" in df.columns:
    ws18 = wb.create_sheet("📋 Fraud Txn Wide Table"); ws18.sheet_view.showGridLines = False
    ws18.freeze_panes = "B1"
    fraud_types_ordered = ["mule_ring","layering","ATO","smurfing","identity_fraud",
                            "dormant_ATO","dormant_smurfing","dormant_to_offshore"]
    ordered_wide  = [k for k in fraud_types_ordered if k in df["fraud_type"].values]
    ordered_wide += [k for k in df[df["label"]==1]["fraud_type"].dropna().unique() if k not in ordered_wide]
    n_cols = len(WIDE_COLS); cur_row = 1

    cur_row = banner(ws18,cur_row,"PART A — 2 Representative Examples Per Fraud Type  (sorted by FIS descending)",n_cols,"1F3864",height=30)
    cur_row = info_row(ws18,cur_row,"Each example is followed by a ⚠️ reasoning row explaining the AML signals for that typology.",n_cols,"BDD7EE",height=26)
    cur_row += 1

    for type_idx, ftype in enumerate(ordered_wide):
        ci_ = type_idx % len(TYPE_COLOURS)
        HDR2_T = PatternFill("solid", start_color=TYPE_HDR2_COLOURS[ci_])
        ALT_T  = PatternFill("solid", start_color=ALT_COLOURS[ci_])
        disp_nm, desc = FRAUD_TYPE_DESCRIPTIONS.get(ftype,(ftype.replace("_"," ").title(),""))
        subset  = df[(df["fraud_type"]==ftype) & (df["label"]==1)].copy()
        sort_by = [c for c in ["fraud_intensity_score","weighted_rule_score","amount"] if c in subset.columns]
        top2    = subset.sort_values(sort_by, ascending=False).head(2)

        cur_row = banner(ws18,cur_row,f"🔴  {disp_nm.upper()}   ({len(subset):,} total transactions)",n_cols,TYPE_COLOURS[ci_])
        cur_row = info_row(ws18,cur_row,desc,n_cols,"FFF2CC",italic=True,height=36)
        for ci,col_name in enumerate(WIDE_COLS,start=1):
            hdr(ws18,cur_row,ci,col_name.replace("_"," ").title(),size=8,fill=HDR2_T,color="1F2D3D")
        ws18.row_dimensions[cur_row].height=28; cur_row+=1

        for ex_num,(_,ex_row) in enumerate(top2.iterrows(),start=1):
            fill = WHITE if ex_num==1 else ALT_T
            for ci,col in enumerate(WIDE_COLS,start=1):
                val = ex_row[col] if col in ex_row.index else ""
                if isinstance(val,float) and col=="amount": val=round(val,2)
                c=ws18.cell(row=cur_row,column=ci,value=val)
                c.font=Font(name="Arial",size=9,color="1F2D3D"); c.fill=fill
                c.alignment=Alignment(horizontal="left" if ci==1 else "center",vertical="center")
                c.border=THIN
            ws18.row_dimensions[cur_row].height=14; cur_row+=1
            sigs=build_fraud_reasoning(ex_row,ftype)
            cur_row=info_row(ws18,cur_row,f"  ⚠️  Example {ex_num}: "+"  ✦  ".join(sigs),n_cols,"FFE0B2",height=30)
        cur_row+=2

    # Part B
    cur_row+=1
    deep_type = "layering" if "layering" in ordered_wide else ordered_wide[0]
    dd_nm,dd_desc=FRAUD_TYPE_DESCRIPTIONS.get(deep_type,(deep_type.replace("_"," ").title(),""))
    dd_ci=ordered_wide.index(deep_type)%len(TYPE_COLOURS)
    DD_HDR2=PatternFill("solid",start_color=TYPE_HDR2_COLOURS[dd_ci])
    DD_ALT=PatternFill("solid",start_color=ALT_COLOURS[dd_ci])
    dd_df=df[(df["fraud_type"]==deep_type)&(df["label"]==1)][WIDE_COLS].copy()
    sort_by=[c for c in ["fraud_intensity_score","weighted_rule_score","amount"] if c in dd_df.columns]
    dd_df=dd_df.sort_values(sort_by,ascending=False).reset_index(drop=True)

    cur_row=banner(ws18,cur_row,f"PART B — Complete Deep-Dive: {dd_nm.upper()}  ({len(dd_df):,} txns · FIS descending)",n_cols,"1F3864",height=30)
    cur_row=info_row(ws18,cur_row,f"All {len(dd_df):,} {dd_nm} fraud transactions, full column set, sorted by FIS.",n_cols,"BDD7EE",height=26)
    for ci,col_name in enumerate(WIDE_COLS,start=1):
        hdr(ws18,cur_row,ci,col_name.replace("_"," ").title(),size=8,fill=DD_HDR2,color="1F2D3D")
    ws18.row_dimensions[cur_row].height=28; cur_row+=1
    for ri,(_,row_data) in enumerate(dd_df.iterrows()):
        fill=DD_ALT if ri%2==1 else WHITE
        for ci,col in enumerate(WIDE_COLS,start=1):
            val=row_data[col] if col in row_data.index else ""
            c=ws18.cell(row=cur_row,column=ci,value=val)
            c.font=Font(name="Arial",size=9,color="1F2D3D"); c.fill=fill
            c.alignment=Alignment(horizontal="left" if ci==1 else "center",vertical="center")
            c.border=THIN
        ws18.row_dimensions[cur_row].height=14; cur_row+=1
    for ci,col in enumerate(WIDE_COLS,start=1):
        cl=get_column_letter(ci)
        sample=[col.replace("_"," ").title()]+[str(ws18.cell(r,ci).value or "") for r in range(1,min(cur_row,100))]
        ws18.column_dimensions[cl].width=min(max(max(len(v) for v in sample)+3,12),40)

    print(f"✅ Sheet 19: Fraud Txn Wide Table ({len(ordered_wide)} types + {len(dd_df):,} {deep_type} rows)")

# ── Save ──────────────────────────────────────────────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_PATH)
print(f"\n✅ Saved → {OUTPUT_PATH}")
print(f"   Sheets ({len(wb.worksheets)}): {[ws.title for ws in wb.worksheets]}")